# Optimization: From Gradient Descent to Adam

Notebook for **Chapter 14** of the *math* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/math/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 What Optimization Is, and Why Convexity Makes It Easy

In [ ]:
import numpy as np

def f(x):  return x**4 - 3*x**2 + x
def df(x): return 4*x**3 - 6*x + 1

for x0 in (-2.0, 2.0):
    x = x0
    for _ in range(200):
        x = x - 0.01 * df(x)             # plain gradient descent, step 0.01
    print(f"start {x0:+.1f} -> x = {x:+.4f}, f(x) = {f(x):+.4f}")

start -2.0 -> x = -1.3008, f(x) = -3.5139
start +2.0 -> x = +1.1309, f(x) = -1.0702


In [ ]:
import numpy as np

# Hessians of three quadratic functions f(x, y) = 0.5 * [x y] H [x y]^T
mats = {
    "x^2 + x*y + y^2": np.array([[2.0, 1.0], [1.0, 2.0]]),
    "x^2 + 3*x*y + y^2": np.array([[2.0, 3.0], [3.0, 2.0]]),
    "x^2 + y^2": np.array([[2.0, 0.0], [0.0, 2.0]]),
}
for name, H in mats.items():
    eig = np.linalg.eigvalsh(H)          # eigenvalues of a symmetric matrix
    verdict = "convex" if eig.min() >= 0 else "NOT convex"
    print(f"{name:28s} eigenvalues {np.round(eig, 3)} -> {verdict}")

x^2 + x*y + y^2              eigenvalues [1. 3.] -> convex
x^2 + 3*x*y + y^2            eigenvalues [-1.  5.] -> NOT convex
x^2 + y^2                    eigenvalues [2. 2.] -> convex


## 14.2 Gradient Descent, Learning Rates and Conditioning

In [ ]:
# Gradient descent on f(x) = x^2 with different learning rates
def run(alpha, x0=4.0, steps=5):
    xs = [x0]
    for _ in range(steps):
        xs.append(xs[-1] - alpha * 2 * xs[-1])   # f'(x) = 2x
    return xs

for alpha in (0.1, 0.5, 1.0, 1.1):
    print(f"alpha={alpha}:", [round(v, 4) for v in run(alpha)])

alpha=0.1: [4.0, 3.2, 2.56, 2.048, 1.6384, 1.3107]
alpha=0.5: [4.0, 0.0, 0.0, 0.0, 0.0, 0.0]
alpha=1.0: [4.0, -4.0, 4.0, -4.0, 4.0, -4.0]
alpha=1.1: [4.0, -4.8, 5.76, -6.912, 8.2944, -9.9533]


In [ ]:
import numpy as np

# f(x, y) = 0.5 * (x^2 + 10 y^2), gradient = (x, 10 y)
def grad(p): return np.array([p[0], 10 * p[1]])

p = np.array([4.0, 2.0])
alpha = 0.15
print(0, p)
for k in range(1, 6):
    p = p - alpha * grad(p)
    print(k, np.round(p, 4))

0 [4. 2.]
1 [ 3.4 -1. ]
2 [2.89 0.5 ]
3 [ 2.4565 -0.25  ]
4 [2.088 0.125]
5 [ 1.7748 -0.0625]


In [ ]:
import numpy as np

# Quadratic with Hessian eigenvalues mu (smallest) and L (largest)
for mu, L in ((1.0, 10.0), (1.0, 100.0), (1.0, 1000.0)):
    kappa = L / mu
    alpha = 2 / (mu + L)                          # best fixed step
    rho = (kappa - 1) / (kappa + 1)               # per-step shrink factor
    steps = np.log(1e-6) / np.log(rho)            # steps to shrink error 1,000,000 times
    print(f"kappa={kappa:6.0f}  rho={rho:.5f}  steps for 1e-6: {steps:8.1f}")

kappa=    10  rho=0.81818  steps for 1e-6:     68.8
kappa=   100  rho=0.98020  steps for 1e-6:    690.8
kappa=  1000  rho=0.99800  steps for 1e-6:   6907.8


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
size = rng.uniform(500, 3000, 200)         # square feet
rooms = rng.integers(1, 6, 200).astype(float)
X = np.column_stack([size, rooms])

def condition(X):
    H = X.T @ X / len(X)                   # Hessian of mean squared error is 2 * this
    e = np.linalg.eigvalsh(H)
    return e.max() / e.min()

Xs = (X - X.mean(axis=0)) / X.std(axis=0)  # standardise: mean 0, std 1
print("condition number raw:         ", round(condition(X), 1))
print("condition number standardised:", round(condition(Xs), 3))

condition number raw:          1334566.1
condition number standardised: 1.259


## 14.3 Batch, Stochastic and Mini-Batch Gradient Descent

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n = 10_000
X = np.column_stack([np.ones(n), rng.normal(size=n)])     # intercept + one feature
true_w = np.array([2.0, 3.0])
y = X @ true_w + rng.normal(scale=1.0, size=n)            # noise std 1

def loss(w):      return np.mean((X @ w - y) ** 2)
def grad(w, idx): # gradient of the mean squared error on rows idx
    Xb, yb = X[idx], y[idx]
    return 2 * Xb.T @ (Xb @ w - yb) / len(idx)

def train(batch, alpha, steps):
    w = np.zeros(2)
    for _ in range(steps):
        idx = rng.choice(n, size=batch, replace=False) if batch < n else np.arange(n)
        w = w - alpha * grad(w, idx)
    return w

# same budget of 20,000 row-gradients per run
for name, batch, steps in (("full batch", n, 2), ("mini-batch 100", 100, 200), ("stochastic (1)", 1, 20000)):
    w = train(batch, 0.1 if batch > 1 else 0.01, steps)
    print(f"{name:15s} steps={steps:6d} w={np.round(w, 3)} loss={loss(w):.4f}")

full batch      steps=     2 w=[0.705 1.082] loss=6.3644
mini-batch 100  steps=   200 w=[1.968 3.078] loss=0.9808
stochastic (1)  steps= 20000 w=[2.055 3.175] loss=1.0027


## 14.4 Momentum, AdaGrad, RMSProp, Adam and AdamW

In [ ]:
# Heavy-ball momentum on f(x) = x^2, alpha = 0.1, beta = 0.9
x, v = 4.0, 0.0
alpha, beta = 0.1, 0.9
for k in range(1, 5):
    g = 2 * x
    v = beta * v + g
    x = x - alpha * v
    print(f"step {k}: g={g:.4f} v={v:.4f} x={x:.4f}")

step 1: g=8.0000 v=8.0000 x=3.2000
step 2: g=6.4000 v=13.6000 x=1.8400
step 3: g=3.6800 v=15.9200 x=0.2480
step 4: g=0.4960 v=14.8240 x=-1.2344


In [ ]:
import numpy as np

# Adam by hand on f(x) = x^2 (gradient 2x), start x = 4
alpha, b1, b2, eps = 0.1, 0.9, 0.999, 1e-8
x, m, v = 4.0, 0.0, 0.0
for t in range(1, 4):
    g = 2 * x
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    x = x - alpha * m_hat / (np.sqrt(v_hat) + eps)
    print(f"t={t} g={g:.4f} m={m:.4f} v={v:.6f} m_hat={m_hat:.4f} v_hat={v_hat:.4f} x={x:.6f}")

t=1 g=8.0000 m=0.8000 v=0.064000 m_hat=8.0000 v_hat=64.0000 x=3.900000
t=2 g=7.8000 m=1.5000 v=0.124776 m_hat=7.8947 v_hat=62.4192 x=3.800074
t=3 g=7.6001 m=2.1100 v=0.182413 m_hat=7.7860 v_hat=60.8653 x=3.700274


In [ ]:
import numpy as np

# ---- from-scratch optimisers: each takes grad(p) and returns the path ----
def gd(grad, p0, alpha, steps):
    p = np.array(p0, float); path = [p.copy()]
    for _ in range(steps):
        p = p - alpha * grad(p); path.append(p.copy())
    return np.array(path)

def momentum(grad, p0, alpha, steps, beta=0.9):
    p = np.array(p0, float); v = np.zeros_like(p); path = [p.copy()]
    for _ in range(steps):
        v = beta * v + grad(p); p = p - alpha * v; path.append(p.copy())
    return np.array(path)

def nesterov(grad, p0, alpha, steps, beta=0.9):
    p = np.array(p0, float); v = np.zeros_like(p); path = [p.copy()]
    for _ in range(steps):
        v = beta * v + grad(p - alpha * beta * v)   # gradient at the look-ahead point
        p = p - alpha * v; path.append(p.copy())
    return np.array(path)

def adagrad(grad, p0, alpha, steps, eps=1e-8):
    p = np.array(p0, float); G = np.zeros_like(p); path = [p.copy()]
    for _ in range(steps):
        g = grad(p); G += g * g
        p = p - alpha * g / (np.sqrt(G) + eps); path.append(p.copy())
    return np.array(path)

def rmsprop(grad, p0, alpha, steps, rho=0.9, eps=1e-8):
    p = np.array(p0, float); s = np.zeros_like(p); path = [p.copy()]
    for _ in range(steps):
        g = grad(p); s = rho * s + (1 - rho) * g * g
        p = p - alpha * g / (np.sqrt(s) + eps); path.append(p.copy())
    return np.array(path)

def adam(grad, p0, alpha, steps, b1=0.9, b2=0.999, eps=1e-8, wd=0.0):
    p = np.array(p0, float); m = np.zeros_like(p); v = np.zeros_like(p); path = [p.copy()]
    for t in range(1, steps + 1):
        g = grad(p)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1 ** t); v_hat = v / (1 - b2 ** t)
        p = p - alpha * (m_hat / (np.sqrt(v_hat) + eps) + wd * p)   # wd > 0 gives AdamW
        path.append(p.copy())
    return np.array(path)

# ---- the ravine: f = 0.5 * (x^2 + 100 y^2), minimum at (0, 0) ----
def ravine(p):  return 0.5 * (p[0] ** 2 + 100 * p[1] ** 2)
def ravine_grad(p): return np.array([p[0], 100 * p[1]])

start = [10.0, 1.0]
runs = {
    "GD (alpha 0.015)":       gd(ravine_grad, start, 0.015, 200),
    "Momentum (0.01, 0.9)":   momentum(ravine_grad, start, 0.01, 200),
    "Nesterov (0.01, 0.9)":   nesterov(ravine_grad, start, 0.01, 200),
    "AdaGrad (2.0)":          adagrad(ravine_grad, start, 2.0, 200),
    "RMSProp (0.05)":         rmsprop(ravine_grad, start, 0.05, 200),
    "Adam (0.2)":             adam(ravine_grad, start, 0.2, 200),
}
for name, path in runs.items():
    print(f"{name:24s} final loss after 200 steps: {ravine(path[-1]):.3e}")

GD (alpha 0.015)         final loss after 200 steps: 1.184e-01
Momentum (0.01, 0.9)     final loss after 200 steps: 1.592e-08
Nesterov (0.01, 0.9)     final loss after 200 steps: 3.229e-09
AdaGrad (2.0)            final loss after 200 steps: 1.448e-18
RMSProp (0.05)           final loss after 200 steps: 2.370e-01
Adam (0.2)               final loss after 200 steps: 4.611e-08


In [ ]:
import numpy as np

def rosen(p):  return (1 - p[0]) ** 2 + 100 * (p[1] - p[0] ** 2) ** 2
def rosen_grad(p):
    return np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0] ** 2),
                     200 * (p[1] - p[0] ** 2)])

def run(update, state, p, steps):
    """Generic loop: update(p, g, state, t) returns the new p and the new state."""
    for t in range(1, steps + 1):
        p, state = update(p, rosen_grad(p), state, t)
    return p

def gd(a):
    return lambda p, g, s, t: (p - a * g, s)

def momentum(a, b=0.9):
    def u(p, g, v, t):
        v = b * v + g
        return p - a * v, v
    return u

def rmsprop(a, rho=0.9, eps=1e-8):
    def u(p, g, s, t):
        s = rho * s + (1 - rho) * g * g
        return p - a * g / (np.sqrt(s) + eps), s
    return u

def adam(a, b1=0.9, b2=0.999, eps=1e-8):
    def u(p, g, st, t):
        m, v = st
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        step = (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)
        return p - a * step, (m, v)
    return u

start = np.array([-1.5, 1.5]); z = np.zeros(2)
for name, upd, st in (("GD 0.001", gd(0.001), None), ("Momentum 0.0002", momentum(0.0002), z),
                      ("RMSProp 0.005", rmsprop(0.005), z), ("Adam 0.05", adam(0.05), (z, z))):
    p = run(upd, st, start, 5000)
    print(f"{name:16s} end at {np.round(p, 3)}  loss {rosen(p):.6f}")

GD 0.001         end at [0.93  0.864]  loss 0.004951
Momentum 0.0002  end at [0.992 0.984]  loss 0.000061
RMSProp 0.005    end at [0.997 0.986]  loss 0.005602
Adam 0.05        end at [1. 1.]  loss 0.000000


In [ ]:
import numpy as np

lr_max, lr_min, T, warm = 0.1, 0.001, 100, 10

def step_decay(t):  return lr_max * 0.5 ** (t // 30)          # halve every 30 steps
def cosine(t):      return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * t / T))
def warmup_cosine(t):
    if t < warm: return lr_max * (t + 1) / warm               # linear ramp up
    progress = (t - warm) / (T - warm)                        # 0 to 1 over the rest
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

print("t    step    cosine  warmup+cosine")
for t in (0, 5, 10, 30, 50, 90, 99):
    print(f"{t:<4d} {step_decay(t):.4f}  {cosine(t):.4f}  {warmup_cosine(t):.4f}")

t    step    cosine  warmup+cosine
0    0.1000  0.1000  0.0100
5    0.1000  0.0994  0.0600
10   0.1000  0.0976  0.1000
30   0.0500  0.0796  0.0884
50   0.0500  0.0505  0.0591
90   0.0125  0.0034  0.0040
99   0.0125  0.0010  0.0010


In [ ]:
import numpy as np

def clip_by_norm(g, c):
    norm = np.linalg.norm(g)
    return g * min(1.0, c / norm)        # shrink only if the norm is above c

g = np.array([30.0, 40.0])               # norm = 50
print("clipped to 5  :", clip_by_norm(g, 5.0))
print("small gradient:", clip_by_norm(np.array([0.3, 0.4]), 5.0))

clipped to 5  : [3. 4.]
small gradient: [0.3 0.4]


In [ ]:
import numpy as np

# f(x, y) = x^2 - y^2 has a saddle at (0, 0); gradient = (2x, -2y)
def grad(p): return np.array([2 * p[0], -2 * p[1]])

for y0 in (0.0, 1e-6):
    p = np.array([1.0, y0]); alpha = 0.1
    for k in range(100):
        p = p - alpha * grad(p)
    print(f"start y = {y0:g}: after 100 steps p = ({p[0]:.2e}, {p[1]:.2e})")

start y = 0: after 100 steps p = (2.04e-10, 0.00e+00)
start y = 1e-06: after 100 steps p = (2.04e-10, 8.28e+01)


## 14.5 Second-Order Methods: Newton, BFGS and Line Search

In [ ]:
import numpy as np

# Minimise f(x) = e^x - 2x.  f'(x) = e^x - 2, f''(x) = e^x.  True minimum: x = ln 2
x = 0.0
print("true answer", round(np.log(2), 10))
for k in range(1, 6):
    x = x - (np.exp(x) - 2) / np.exp(x)       # Newton step: x - f'/f''
    print(f"step {k}: x = {x:.10f}  error = {abs(x - np.log(2)):.2e}")

true answer 0.6931471806
step 1: x = 1.0000000000  error = 3.07e-01
step 2: x = 0.7357588823  error = 4.26e-02
step 3: x = 0.6940422999  error = 8.95e-04
step 4: x = 0.6931475811  error = 4.00e-07
step 5: x = 0.6931471806  error = 8.02e-14


In [ ]:
import numpy as np

# f(x, y) = x^2 + 10 y^2: one Newton step lands on the minimum
H = np.array([[2.0, 0.0], [0.0, 20.0]])
p = np.array([4.0, 2.0])
g = np.array([2 * p[0], 20 * p[1]])
step = np.linalg.solve(H, g)                  # solve H s = g, never form H^-1
print("gradient", g, " Newton step", step, " new point", p - step)

gradient [ 8. 40.]  Newton step [4. 2.]  new point [0. 0.]


In [ ]:
import numpy as np
from scipy.optimize import minimize

def rosen(p):  return (1 - p[0]) ** 2 + 100 * (p[1] - p[0] ** 2) ** 2
def rosen_grad(p):
    return np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0] ** 2),
                     200 * (p[1] - p[0] ** 2)])

for method in ("BFGS", "L-BFGS-B", "Newton-CG"):
    r = minimize(rosen, [-1.5, 1.5], jac=rosen_grad, method=method)
    print(f"{method:10s} x = {np.round(r.x, 4)}  iterations = {r.nit}  f evaluations = {r.nfev}")

BFGS       x = [1. 1.]  iterations = 34  f evaluations = 41
L-BFGS-B   x = [1. 1.]  iterations = 40  f evaluations = 49
Newton-CG  x = [1. 1.]  iterations = 56  f evaluations = 74


In [ ]:
# Backtracking line search (Armijo) on f(x) = x^2 from x = 4
f = lambda x: x * x
x, g = 4.0, 8.0
c, alpha = 0.1, 1.0
while f(x - alpha * g) > f(x) - c * alpha * g * g:      # not enough decrease yet
    print(f"alpha={alpha}: f(new)={f(x - alpha * g):.2f} > bound {f(x) - c * alpha * g * g:.2f}, halve")
    alpha /= 2
print(f"accepted alpha={alpha}: f(new)={f(x - alpha * g):.2f} <= bound {f(x) - c * alpha * g * g:.2f}")

alpha=1.0: f(new)=16.00 > bound 9.60, halve
accepted alpha=0.5: f(new)=0.00 <= bound 12.80


## 14.6 Constrained Optimization, KKT, Duality and Regularization

In [ ]:
import numpy as np
from scipy.optimize import minimize

# minimise (x-2)^2 + (y-2)^2  subject to  x + y <= 2
obj = lambda p: (p[0] - 2) ** 2 + (p[1] - 2) ** 2
con = {"type": "ineq", "fun": lambda p: 2 - p[0] - p[1]}      # scipy wants g(p) >= 0
r = minimize(obj, [0.0, 0.0], constraints=[con], method="SLSQP")
x, y = r.x
mu = -2 * (x - 2)                                             # from 2(x-2) + mu = 0
print("solution", np.round(r.x, 4), "objective", round(r.fun, 4))
print("multiplier mu =", round(mu, 4), " slack 2-x-y =", round(2 - x - y, 6))

solution [1. 1.] objective 2.0
multiplier mu = 2.0  slack 2-x-y = 0.0


In [ ]:
import numpy as np

rng = np.random.default_rng(3)
n = 50
x1 = rng.normal(size=n)
x2 = x1 + rng.normal(scale=0.05, size=n)         # almost a copy of x1
X = np.column_stack([x1, x2])
y = 3 * x1 + rng.normal(scale=0.5, size=n)

def ridge(X, y, lam):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)   # (X^T X + lam I)^-1 X^T y

for lam in (0.0, 1.0, 10.0):
    w = ridge(X, y, lam)
    print(f"lambda={lam:5.1f} w={np.round(w, 3)}  sum={w.sum():.3f}  ||w||={np.linalg.norm(w):.3f}")

lambda=  0.0 w=[2.109 0.932]  sum=3.042  ||w||=2.306
lambda=  1.0 w=[1.539 1.474]  sum=3.013  ||w||=2.131
lambda= 10.0 w=[1.402 1.403]  sum=2.804  ||w||=1.983


In [ ]:
import numpy as np

def lasso_1d(a, lam): return np.sign(a) * max(abs(a) - lam, 0.0)   # soft threshold
def ridge_1d(a, lam): return a / (1 + lam)

for a in (3.0, 0.5, -2.0):
    print(f"a={a:+.1f}  lasso(lam=1)={lasso_1d(a, 1.0):+.3f}  ridge(lam=1)={ridge_1d(a, 1.0):+.3f}")

a=+3.0  lasso(lam=1)=+2.000  ridge(lam=1)=+1.500
a=+0.5  lasso(lam=1)=+0.000  ridge(lam=1)=+0.250
a=-2.0  lasso(lam=1)=-1.000  ridge(lam=1)=-1.000


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n, d = 100, 10
X = rng.normal(size=(n, d))
true_w = np.zeros(d); true_w[[1, 4, 7]] = [3.0, -2.0, 1.5]       # only 3 real features
y = X @ true_w + rng.normal(scale=0.5, size=n)

def soft(z, t): return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)

lam = 0.2
L = np.linalg.eigvalsh(X.T @ X / n).max()                         # smoothness of the data term
alpha = 1 / L
w = np.zeros(d)
for k in range(500):
    g = X.T @ (X @ w - y) / n                                     # gradient of 0.5 * mean squared error
    w = soft(w - alpha * g, alpha * lam)                          # gradient step, then shrink
print("ISTA  :", np.round(w, 2))
print("OLS   :", np.round(np.linalg.lstsq(X, y, rcond=None)[0], 2))

ISTA  : [ 0.    2.7  -0.   -0.   -1.7   0.    0.    1.21  0.    0.  ]
OLS   : [ 0.1   2.92 -0.05 -0.02 -2.   -0.05 -0.04  1.54  0.01  0.05]


## 14.7 Linear Programming, Integer Programming, Hyperparameter Search and Derivative-Free Methods

In [ ]:
from scipy.optimize import linprog

# trays of croissants (x) and loaves (y): max 40x + 30y
# oven: 2x + y <= 100 hours, labour: x + 3y <= 120 hours
res = linprog(c=[-40, -30],                       # linprog minimises, so negate the profit
              A_ub=[[2, 1], [1, 3]], b_ub=[100, 120],
              bounds=[(0, None), (0, None)], method="highs")
print("plan:", res.x, " profit:", -res.fun)
print("shadow prices (profit per extra hour):", -res.ineqlin.marginals)

plan: [36. 28.]  profit: 2280.0
shadow prices (profit per extra hour): [18.  4.]


In [ ]:
import numpy as np
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

# max 5x + 4y  s.t.  6x + 4y <= 24,  x + 2y <= 6,  x, y >= 0
c = np.array([-5.0, -4.0]); A = np.array([[6.0, 4.0], [1.0, 2.0]]); b = np.array([24.0, 6.0])
lp = linprog(c, A_ub=A, b_ub=b, bounds=[(0, None)] * 2, method="highs")
ip = milp(c, constraints=LinearConstraint(A, ub=b), integrality=np.ones(2), bounds=Bounds(0, np.inf))
print("LP  :", lp.x, -lp.fun)
print("MILP:", np.round(ip.x) + 0.0, -ip.fun)
best = max((5*x + 4*y, x, y) for x in range(5) for y in range(4) if 6*x + 4*y <= 24 and x + 2*y <= 6)
print("brute force:", best)

LP  : [3.  1.5] 21.0
MILP: [4. 0.] 20.0
brute force: (20, 4, 0)


In [ ]:
import numpy as np

# Toy validation score: only the learning rate matters, best at 0.37; the other knob is noise
def score(a, b): return -(a - 0.37) ** 2

rng = np.random.default_rng(0)
grid_best, rand_best = [], []
for trial in range(2000):
    # random shift so the optimum is not always lined up with the grid
    shift = rng.uniform(-0.1, 0.1)
    s = lambda a, b: score(a - shift, b)
    g = [s(a, b) for a in (0.1, 0.5, 0.9) for b in (0.1, 0.5, 0.9)]        # 3 x 3 grid = 9 runs
    r = [s(rng.uniform(0, 1), rng.uniform(0, 1)) for _ in range(9)]       # 9 random runs
    grid_best.append(max(g)); rand_best.append(max(r))
print("average best score, grid  :", round(float(np.mean(grid_best)), 5))
print("average best score, random:", round(float(np.mean(rand_best)), 5))

average best score, grid  : -0.01824
average best score, random: -0.0045


In [ ]:
import numpy as np
from scipy.optimize import minimize

def rosen(p): return (1 - p[0]) ** 2 + 100 * (p[1] - p[0] ** 2) ** 2
r = minimize(rosen, [-1.5, 1.5], method="Nelder-Mead", options={"xatol": 1e-8, "fatol": 1e-10})
print("Nelder-Mead:", np.round(r.x, 4), " evaluations:", r.nfev)

# simulated annealing on a bumpy 1-D function with many local minima
f = lambda x: x ** 2 + 10 * np.sin(3 * x)
rng = np.random.default_rng(0)
x = 4.0; best = x
for k in range(3000):
    T = 5.0 * (1 - k / 3000) + 1e-3                       # temperature falls to ~0
    cand = x + rng.normal(scale=0.5)
    if f(cand) < f(x) or rng.random() < np.exp((f(x) - f(cand)) / T):
        x = cand                                           # sometimes accept a worse point
    if f(x) < f(best): best = x
grid = np.linspace(-6, 6, 120001)
print("annealing best x:", round(best, 3), " f:", round(f(best), 3))
print("grid search  x  :", round(grid[np.argmin(f(grid))], 3), " f:", round(f(grid).min(), 3))

Nelder-Mead: [1. 1.]  evaluations: 252
annealing best x: -0.512  f: -9.732
grid search  x  : -0.512  f: -9.732


## 14.8 The Expert Layer: Rates, Smoothness, Proximal Methods and Flat Minima

In [ ]:
import numpy as np

# Polyak step size alpha = (f(x) - f*) / ||g||^2 on f(x) = x^2 (f* = 0), x0 = 4
x = 4.0
for k in range(1, 5):
    g = 2 * x
    alpha = (x * x - 0.0) / g ** 2
    x = x - alpha * g
    print(f"step {k}: alpha={alpha:.3f} x={x:.4f}")

# Rate comparison on a quadratic with kappa = 100
kappa = 100
rho_gd = (kappa - 1) / (kappa + 1)
rho_hb = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
print("GD   rho", round(rho_gd, 4), " steps for 1e-6:", round(np.log(1e-6) / np.log(rho_gd)))
print("Mom. rho", round(rho_hb, 4), " steps for 1e-6:", round(np.log(1e-6) / np.log(rho_hb)))

step 1: alpha=0.250 x=2.0000
step 2: alpha=0.250 x=1.0000
step 3: alpha=0.250 x=0.5000
step 4: alpha=0.250 x=0.2500
GD   rho 0.9802  steps for 1e-6: 691
Mom. rho 0.8182  steps for 1e-6: 69


## Chapter recap

## Practice set

Problem 14.9.1 Run three steps of gradient descent on f(x) = (x − 3)² from x0 = 0 with α = 0.25.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/math/ch14.html